In [58]:
import load_mails
import importlib
importlib.reload(load_mails)


service = load_mails.get_gmail_service()
emails = load_mails.fetch_unread_emails(service)

Searching for unread mails...
Found 35 unread mails :)
Email #1
From    : "Asfar Butt ." <2503599@students.au.edu.pk>
Subject : Wrong item received
Date    : Sun, 9 Aug 2026 12:50:07 -0700

Body Preview:
  Hi, I ordered a wireless mouse but got a keyboard instead. Just wanted to
flag this and see how we can get it sorted out.



Email #2
From    : Mail Delivery Subsystem <mailer-daemon@googlemail.com>
Subject : Delivery Status Notification (Failure)
Date    : Sun, 09 Aug 2026 12:40:08 -0700 (PDT)

Body Preview:

** Address not found **

Your message wasn't delivered to christopher.graves73755@hotmail.com because the address couldn't be found, or is unable to receive mail.



The response from the remote server was:
550 5.5.0 Requested action not taken: mailbox unavailable (S2017062302). [BL6PEPF00022574.namprd02.prod.outlook.com 2026-08-09T19:40:08.931Z 08DEF30102D76189]



Email #3
From    : Mail Delivery Subsystem <mailer-daemon@googlemail.com>
Subject : Delivery Status Notification 

In [37]:
email = emails[0]
print(email)

{'id': '19fe80d6c0fa61b2', 'thread_id': '19fe80d6c0fa61b2', 'sender': '"Asfar Butt ." <2503599@students.au.edu.pk>', 'subject': 'My order arrived damaged', 'date': 'Sun, 9 Aug 2026 12:43:13 -0700', 'body': "Hi, I received my order today but the box was pretty banged up and the item\r\ninside doesn't look right either — think it might be broken. Not sure what\r\nto do from here, can you help?\r\n"}


In [52]:
email = {'id': '19fe80d6c0fa61b2', 'thread_id': '19fe80d6c0fa61b2', 'sender': '2503599@students.au.edu.pk', 'subject': 'Problem with my laptop order', 'date': 'Sun, 9 Aug 2026 12:43:13 -0700', 'body': "Hi, I bought the Rangeforge Laptop G3688 a little while back and it's been overheating like crazy, even with barely anything running. Getting worried it might be a hardware issue. Can you let me know what my options are?"}


In [57]:
import pandas
abc = pandas.read_csv('../assets/orders.csv.gz')
print(abc[abc['customer_id'] == 'CUST0422521'])

           order_id  customer_id  product_id  order_date      status  \
5754   ORD000685187  CUST0422521  PROD000076  2024-10-12  in_transit   
6722   ORD000798907  CUST0422521  PROD099407  2023-09-07    returned   
14115  ORD001636337  CUST0422521  PROD064967  2025-01-30   delivered   
32294  ORD003794236  CUST0422521  PROD037977  2026-07-28    returned   

        tracking_id warehouse_id  
5754   TRK000685187        WH021  
6722   TRK000798907        WH019  
14115  TRK001636337        WH015  
32294  TRK003794236        WH035  


In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

In [31]:
# email = {'id': '19fdc54642d89380', 'thread_id': '19fdc54642d89380', 'sender': 'christopher.graves73755@hotmail.com', 'subject': 'where is my order', 'date': 'Fri, 7 Aug 2026 06:05:17 -0700', 'body': "So I ordered something like 2 weeks ago and it's just... not what I expected honestly. Kind of annoyed. What are my options here?"}

In [45]:
# User Verification

import importlib
import customer_verification
importlib.reload(customer_verification)

response = customer_verification.user_verification(email)
print(response)
print("\n\nVerification successful" if response["verification"] else "\n\nVerification unsuccessful")

Complaint Summary:
I bought a Rangeforge Laptop G3688 that's been overheating excessively, even with minimal usage, and I'm concerned it might be a hardware issue.
Customer already provided verification information.
{'order_id': None, 'customer_id': None, 'email': 'carol.franco12997@hotmail.com', 'user_name': '', 'email_summary': "I bought a Rangeforge Laptop G3688 that's been overheating excessively, even with minimal usage, and I'm concerned it might be a hardware issue.", 'verification': True}


Verification successful


In [46]:
import dataset_retrieval
importlib.reload(dataset_retrieval)

cell2_response = {"success":False}
def check_id(response):
    if response['verification']:
        if response['order_id'] is not None:
            print("OrderID check")
            ans = dataset_retrieval.fetch_data('','order_id',response['order_id'])
            if ans == None:
                return
            cell2_response['success'] = True
            cell2_response['order_id'] = ans[0]
            cell2_response['customer_id'] = ans[1]
            cell2_response['product_id'] = ans[2]
            print("Order found")
            
        elif response['customer_id'] is not None:
            print("CustomerID check")
            ans = dataset_retrieval.fetch_data('','customer_id',response['customer_id'])
            if ans == None:
                return
            cell2_response['success'] = True
            cell2_response['customer_id'] = ans[0]
            ans = dataset_retrieval.fetch_data('orders', 'customer_id', response['customer_id'])
            if ans == None:
                return
            cell2_response['order_id'] = ans[0]
            cell2_response['product_id'] = ans[2]
        elif response['email'] is not None:
            print("Email check")
            ans = dataset_retrieval.fetch_data('customers','email',response['email'])
            if ans == None:
                return 
            cell2_response['success'] = True
            cell2_response['customer_id'] = ans[0]
            ans = dataset_retrieval.fetch_data('orders', 'customer_id', cell2_response['customer_id'])
            if ans == None:
                return
            cell2_response['order_id'] = ans[0] 
            cell2_response['product_id'] = ans[2]
    else:
        print("Nothing to check against")

check_id(response)
print(cell2_response)

if cell2_response['success']:
    cell2_response['embed_query'] = email['body']



('PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit',)
('ORD000000262', 'CUST0459416', 'PROD100170', datetime.date(2024, 9, 22), 'delivered', 'TRK000000262', 'WH046')
Email check
('PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit',)
('PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit',)
{'success': True, 'customer_id': 'CUST0422521', 'order_id': 'ORD000685187', 'product_id': 'PROD000076'}


In [47]:
import email_reply
importlib.reload(email_reply)

if not cell2_response['success']:
    apology_email01 = {
        "sender": email['sender'],
        "subject": "Re: " + email['subject'],
        "body": (
            f"Hi {response['user_name']},\n\n"
            "We ran into an issue verifying your details on our end. To move forward, "
            "could you confirm a few things — your order ID, the email used at checkout, "
            "or any other reference that ties back to your purchase?\n\n"
            "Once we've got that, we'll pick this right back up.\n\n"
            "Best regards,\nTechSphere Support Team"
        ),
        "id": email['id'],
        "thread_id": email['thread_id'],
    }

    print_ans01 = email_reply.mail_sender(
        apology_email01["sender"],
        apology_email01["subject"], 
        apology_email01["body"],
        apology_email01['id'],
        apology_email01["thread_id"], 
        "customer_not_found"
    )

    print(print_ans01)

## User Verified
`cell2_response['success']` contain its details

In [48]:
import embedding_retrieval
importlib.reload(embedding_retrieval)

embed_response_docs = embedding_retrieval.get_embedding_context(cell2_response['embed_query'])

print(embed_response_docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6126.54it/s]


No relevant documents found.
No relevant documents found.


In [49]:
order = dataset_retrieval.fetch_data('','product_id',cell2_response['product_id'])
order_details = [order[0],order[1], order[2], order[5]]

('PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit',)


In [53]:
# check_policy
# retry_retrieval
# ask_customer
# resolve
# escalate

# mail_body = "So I ordered something like 2 weeks ago and it's just... not what I expected honestly. Kind of annoyed. What are my options here?"

from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="E:/Asfar/Learning/Project01/codefiles/.env")

prompt01 = ChatPromptTemplate.from_template("""
You are drafting a response to a verified customer complaint for TechSphere. You have confirmed order and customer information available in context.
User input: {mail_body}
User order details: {order_details}
Retrieved docs: {docs}
Use confidential content only to inform your reasoning — never quote, reference, or reveal it to the customer. Public content may be referenced or paraphrased.
Match the customer's tone and energy from their message. Read {mail_body} for register (casual vs formal), emotional temperature (annoyed, neutral, worried), and phrasing style — then write in a comparable register. Don't upgrade a casual, slightly annoyed email into a stiff corporate tone, and don't downgrade a formal email into forced casualness. Mirror their level of directness too: if they were brief, don't pad; if they explained things at length, it's fine to be a bit more conversational. Stay warm and professional even if the customer is heated — match their energy, not their rudeness.
Decide: are these docs sufficient to write a specific, policy-grounded reply to this exact complaint? Generic reassurance without specific grounding does not count as sufficient.
Choose exactly one action:
- "respond" — docs are sufficient. Write the full customer-facing reply: specific, helpful, discreet. Never mention internal documents, policies by name, or that you searched anything.
- "retry_retrieval" — docs are not sufficient. Write a refined search query, more specific than the original, targeting exactly what's missing.
Return ONLY this JSON, nothing else:
{{
  "action": "respond" or "retry_retrieval",
  "response": string (only if action is "respond", else ""),
  "refined_query": string (only if action is "retry_retrieval", else "")
}}
""")

prompt02 = ChatPromptTemplate.from_template("""
You are drafting a response to a verified customer complaint for TechSphere. This is the final attempt — retrieval cannot be retried again.
User input: {mail_body}
User order details: {order_details}
Retrieved docs (original + refined): {docs}
Use confidential content only to inform your reasoning — never quote, reference, or reveal it to the customer. Public content may be referenced or paraphrased.
Match the customer's tone and energy from their message. Read {mail_body} for register (casual vs formal), emotional temperature (annoyed, neutral, worried), and phrasing style — then write in a comparable register. Don't upgrade a casual, slightly annoyed email into a stiff corporate tone, and don't downgrade a formal email into forced casualness. Mirror their level of directness too: if they were brief, don't pad; if they explained things at length, it's fine to be a bit more conversational. Stay warm and professional even if the customer is heated — match their energy, not their rudeness.
Choose exactly one action:
- "respond" — docs are sufficient for a specific, policy-grounded reply. Write it: specific, helpful, discreet. Never mention internal documents, policies by name, or that you searched anything.
- "ask_customer" — docs are close, but the reply depends on a detail the customer hasn't given (which item, condition on arrival, refund vs exchange preference, etc). Ask only for that missing detail — no generic "tell me more."
- "escalate" — docs are genuinely insufficient and no customer-provided detail would fix that (policy gap, uncovered case). Write nothing customer-facing. Write a short internal note for the case owner explaining exactly what's missing.
Return ONLY this JSON, nothing else:
{{
  "action": "respond" or "ask_customer" or "escalate",
  "response": string (only if action is "respond", else ""),
  "customer_message": string (only if action is "ask_customer", else ""),
  "escalation_reason": string (only if action is "escalate", else "")
}}
""")

GROQ_API_KEY02 = os.getenv("GROQ_API_KEY02")

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = GROQ_API_KEY02
)

def drafting_response_mail(docs, mail_body, order_details):
  chain01 = prompt01 | llm | JsonOutputParser()

  cell3_response = chain01.invoke({'mail_body':mail_body, 
                      'docs':docs,
                      'order_details':order_details})

  print(cell3_response)

  if cell3_response['action'] == 'respond':
    return cell3_response

  if cell3_response['action'] == 'retry_retrieval' and cell3_response['refined_query'] != '':
    docs = embedding_retrieval.get_embedding_context(cell3_response['refined_query'])
    print('Retying with refined docs: ',docs)
    chain02 = prompt02 | llm | JsonOutputParser()
    cell3_response = chain02.invoke({'mail_body':mail_body, 
                    'docs':docs,
                    'order_details':order_details})
    print(cell3_response)

    return cell3_response


cell3_response = drafting_response_mail(embed_response_docs, email['body'], order_details)


{'action': 'retry_retrieval', 'response': '', 'refined_query': 'Rangeforge Laptop G3688 overheating issue troubleshooting and warranty options'}
No relevant documents found.
Retying with refined docs:  No relevant documents found.
{'action': 'ask_customer', 'response': '', 'customer_message': "I'm concerned to hear that your Rangeforge Laptop G3688 is overheating. However, I noticed that the order details we have on file are for a Cortexa Router X6793. Could you please confirm if you have indeed purchased the Rangeforge Laptop G3688 from us and provide the order number or details of that purchase?", 'escalation_reason': ''}


In [54]:
import email_reply
importlib.reload(email_reply)

def take_action(response):
    cell4_response = ""
    if response['action'] == 'respond' and response['response'] != '':
        print("Query resolved from our end")
        cell4_response = email_reply.mail_sender(
        email["sender"],
        'Re: Your Recent Query', 
        response['response'],
        email['id'],
        email["thread_id"],
        "case_closed"
        )

    elif response['action'] == 'ask_customer' and response['customer_message'] != '':
        print("Asked the user for more info")
        cell4_response = email_reply.mail_sender(
            email["sender"],
            'Re: Your Recent Query',
            response['customer_message'],
            email['id'],
            email['thread_id'],
            "more_info_needed"
        )

    elif response['action'] == 'escalate' and response['escalation_reason'] != '':
        print("LLM can't figure out the issue to the core.. complaint esclated to host")
        # tell host about it.. he/she will manually do something about it (for rare/special cases)
        None
    return cell4_response

cell4_response = take_action(cell3_response)
print(cell4_response)

Asked the user for more info
Label added successfully :)
{'id': '19fe81c7785325ba', 'threadId': '19fe80d6c0fa61b2', 'labelIds': ['SENT']}
